In [1]:
import math 
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
class Value:
    index = 0
    def __init__(self, data, _children=(), _op='', label=None):
        self.data = data
        if label:
            self.label = label
        else :
            self.label =f"v_{Value.index}"
            Value.index += 1
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set[Value](_children)
        self._op = _op

    def __repr__(self):
        #return f"Value({self.label}: data={self.data}), grad={round(self.grad,10)}, op={self._op}"
        return f"Value({self.label}: data={self.data}), grad={self.grad}, op={self._op}"

    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out =  Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + other*(-1)

    def __rsub__(self, other):
        return self*(-1) + other

    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out        

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float are supported"
        out =  Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += out.grad* other*self.data**(other-1)
        out._backward = _backward
        return out   

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def tanh(self):
        out =  Value(math.tanh(self.data), (self,), 'tanh')
        def _backward():
            self.grad += out.grad/(math.cosh(self.data)**2)
        out._backward = _backward
        return out   

    def exp(self):
        out =  Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.grad*math.exp(self.data)
        out._backward = _backward
        return out   

    def print_chain(self):
        if self._prev:
            for p in self._prev:
                p.print_chain()
        print(self)

    
    def backward(self) -> None:
        self.grad = 1.0

        # topological sort the nodes
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # backward pass
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
                    



In [ ]:
"""
# input layer
a = Value(3.0, label='a')
b = a.tanh()
b.label = "b"
b.backward()
print("--------------------------------")
b.print_chain()


x = Value(3.0, label='x')
tmp = (-1)*x
# y = x.exp() - tmp.exp()  OK
# y = x.exp() + tmp.exp()  OK
# y =(tmp**2)*(tmp**(-1)) OK
y = (x.exp() - tmp.exp()) / (x.exp() + tmp.exp())  # OK
y.label = "y"
y.backward()
print("--------------------------------")
y.print_chain()



a = Value(3.0, label='a')
b = 2*a
b.label = "b"
c = (3*b) - (2*b)
c.label = "c"
c.backward()
print("--------------------------------")
c.print_chain()


a = Value(3.0, label='a')
b = a+a; b.label = "b"
b.backward()
print("--------------------------------")
b.print_chain()

a = Value(-2.0, label='a')
b = Value(3.0, label='b')
d = a*b; d.label = "d"
e = a + b; e.label = "e"
f = d*e; f.label = "f"
f.backward()
print("--------------------------------")
f.print_chain()
"""

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

b = Value(6.8813735870195432, label='b')

n = x1*w1 + x2*w2 + b; n.label = "n"
#o = n.tanh()
mn = -1*n; mn.label = "mn"
o = (n.exp() - mn.exp()) / (n.exp() + mn.exp()); o.label = "o"
o.backward()
print("--------------------------------")
o.print_chain()













SyntaxError: invalid syntax (4190726968.py, line 60)